# **Exploring the Auto-insurance Dataset**
This notebook contains the Exploratory Data Analysis on an Auto-Insurance Fraud Dataset.
## Data Context and Origin
- **Origin:** The dataset was originally curated for a demo found on an [Oracle article on fraud detection through machine learning](https://blogs.oracle.com/coretec/oracle-analytics-oracle-machine-learning-fraud-detection-using-unsupervised-and-supervised-machine-learning)
- **Data Source:** Relative information has been throughly explored from the limited metadata in the [kaggle repository](https://www.kaggle.com/datasets/shivamb/vehicle-claim-fraud-detection/data)
- **Domain Context:** Additional insights and domain-specific context has been formalized, see the [project data section in the domain notes](../../summary/docs/domain_notes.md#the-project-data) document in the repository.
## Analysis Workflow
Subsequent analyses hereon explores:
- [**Data Overview:**](#data-overview) nature of distributions
- [**Feature Insights and Distributions:**](#feature-insights-and-distributions) feature insights
- [**Inspecting Anomalies and Outliers:**](#inspecting-anomalies-and-outliers) outlier analysis
- [**Modeling Preparation:**](#modeling-preparation) pre-emptive processes to prepare the data for modeling
- [**Business Intelligence (BI) Initialization:**](#business-intelligence-(bi)-initialization) initializing a business intelligence data table

Package Imports

In [ ]:
# Data structure package
import pandas as pd 
# Data Treatment package
import numpy as np
from scipy import stats
# Data Visualization package
import matplotlib.pyplot as plt
import seaborn as sns

## Data Overview

In [ ]:
# Data Import
df = pd.read_csv('../../data/raw/fraud_oracle.csv')
# Checking if the data loads correctly
df.head()

An initial view requires inspection of the dataset's loaded datatypes, summary statistics, value counts, as well as checking unique and missing values.
- A function can therefore be defined that will be repetitively used for raw, processed, and final basic inspections. (to be migrated to py modules once initial project can work)

In [ ]:
def eda_summary(df):
    """Generates a DataFrame that summarizes basic inspection for Exploratory Data Analysis (EDA)."""
    # 1. Basic Metadata (current dtype, null/non-null,unique counts)
    summary = pd.DataFrame(
        {
            "dtype": df.dtypes,
            "non_null_count": df.notnull().sum(),
            "missing_count": df.isnull().sum(),
            "n_unique": df.nunique()
        }
    )
    # 2. Summary statistics
    desc = df.describe(include='all').T #transposed describe output

    # 3. Combining metadata and stats
    stat_cols = ['mean','std','min','25%','50%','75%','max']
    summary = summary.join(desc[stat_cols], how='left')
    return summary

In [ ]:
eda_summary(df)

The initial inspection draws attention to the following issues:
- **Data types are str and int64** -> convert appropriately
- **Variables mostly have low unique counts relative to the total data count** -> conversion to categorical or bool types may be an option.
- **PolicyNumber's n_unqiue is equal to the total data count with continuous increment (1 - 15420)** -> can be set as dataframe index as a unique identifier.
- **Age has a peculiar min statistic = 0** -> to cross-examin with AgeofPolicyHolder (bracketed age) before deciding to impute or flag to remove as an outlier.
- **Columns may be redundant for some parts of the workflow** -> feature selection will be done to judge/reason which variables are to be fed to avoid variable co-dependence that can hinder appropriate ML outcomes.

In [ ]:
# PolicyNumber as index
df = df.set_index('PolicyNumber')

In [ ]:
for col in df.select_dtypes(include=['str']).columns:
    print(f"---  Value Counts for: {col} ---")
    print(df[col].value_counts())
    print("\n")

In [ ]:
for col in df:
    if df[col].dtype == 'str':
        df[col] = df[col].astype('category')
    if df[col].dtype == 'int64':
        df[col] = df[col].astype('int32')
df.dtypes

Checking to order categorical columns for future use

In [ ]:
df

Visualizing distributions

In [ ]:
for col in df.select_dtypes(include='category'):
    # Normalized cross-tab 
    crosstab_norm = pd.crosstab(df[col], df['FraudFound_P'], normalize='index') * 100
    # Sorting to have the highest fraud rates at the top left
    if 1 in crosstab_norm.columns:
        crosstab_norm = crosstab_norm.sort_values(by=1, ascending=False)

    # Plotting Crosstab
    crosstab_norm.plot(kind='barh', stacked=True, color=['r','b'], figsize=(8,4))
    # Plotting (countplot)
    #plt.figure(figsize=[8,6])
    #ax = sns.countplot(data=df, y=df[col], hue='FraudFound_P')
    #for container in ax.containers:
    #    ax.bar_label(container, fontsize=9, padding=4)
    #sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))
    #ax.set_xlim(0, ax.get_xlim()[1] * 1.15)
    
    plt.show()
    

Numerical Feature Distributions

## Feature Insights and Distributions

## Inspecting Anomalies and Outliers

## Modeling Preparation

## Business Intelligence (BI) Initialization